In [1]:
import os
import glob
import polars as pl

# ==========================================
# 1. CONFIGURATION PATHS & CONSTANTS
# ==========================================
DATA_DIR = r"F:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds"
TARGET_MAP = "vertigo"
MATCHES_TO_PROCESS = 10

def find_vertigo_bounds():
    print(f"Searching for {MATCHES_TO_PROCESS} matches on map '{TARGET_MAP}'...\n")
    
    # Initialize global boundary tracking variables
    global_min_x = float('inf')
    global_max_x = float('-inf')
    global_min_y = float('inf')
    global_max_y = float('-inf')
    
    matches_processed = 0
    
    # ==========================================
    # 2. FILE DISCOVERY & PARSING
    # ==========================================
    # Find all header files recursively, ignoring exact extensions
    search_pattern = os.path.join(DATA_DIR, "**", "header*")
    header_files = glob.glob(search_pattern, recursive=True)
    
    for header_path in header_files:
        if not os.path.isfile(header_path):
            continue
            
        match_dir = os.path.dirname(header_path)
        match_id = os.path.basename(match_dir)
        
        # Attempt to read the header file to verify the map
        try:
            df_header = pl.read_parquet(header_path)
            raw_map_name = str(df_header["map_name"][0]).lower()
            map_name = raw_map_name.replace("de_", "")  # Normalize 'de_vertigo' to 'vertigo'
        except Exception:
            # Skip corrupted/invalid files silently
            continue
            
        # ==========================================
        # 3. WEAPON FIRE PARSING & BOUNDARY EXTRACTION
        # ==========================================
        if map_name == TARGET_MAP:
            wf_pattern = os.path.join(match_dir, "weapon_fire*")
            wf_files = glob.glob(wf_pattern)
            
            if not wf_files:
                continue
                
            wf_path = wf_files[0]
            
            try:
                df_wf = pl.read_parquet(wf_path)
                
                # Verify columns exist
                if "player_x_pos" in df_wf.columns and "player_y_pos" in df_wf.columns:
                    # Find min/max for this specific match
                    local_min_x = df_wf["player_x_pos"].min()
                    local_max_x = df_wf["player_x_pos"].max()
                    local_min_y = df_wf["player_y_pos"].min()
                    local_max_y = df_wf["player_y_pos"].max()
                    
                    # Update the global map bounds
                    global_min_x = min(global_min_x, local_min_x)
                    global_max_x = max(global_max_x, local_max_x)
                    global_min_y = min(global_min_y, local_min_y)
                    global_max_y = max(global_max_y, local_max_y)
                    
                    matches_processed += 1
                    print(f"[{matches_processed}/{MATCHES_TO_PROCESS}] Processed bounds for match: {match_id}")
                    
                    # Stop once we hit our target sample size
                    if matches_processed >= MATCHES_TO_PROCESS:
                        break
            except Exception as e:
                print(f"Error reading weapon_fire for match {match_id}: {e}")
                continue

    # ==========================================
    # 4. RESULTS DISPLAY
    # ==========================================
    if matches_processed > 0:
        print("\n" + "="*50)
        print(f" EMPIRICAL MAP BOUNDS FOR: {TARGET_MAP.upper()}")
        print(f" Matches analyzed: {matches_processed}")
        print("="*50)
        print(f" x_min : {global_min_x:>10.2f}")
        print(f" x_max : {global_max_x:>10.2f}")
        print(f" y_min : {global_min_y:>10.2f}")
        print(f" y_max : {global_max_y:>10.2f}")
        print("="*50)
        
        # Print dictionary format for easy copy-pasting into your main script
        print("\nDictionary Output format:")
        print(f"MAP_BOUNDS_MIRAGE = {{'x_min': {global_min_x}, 'x_max': {global_max_x}, 'y_min': {global_min_y}, 'y_max': {global_max_y}}}")
    else:
        print(f"No valid matches found for map '{TARGET_MAP}'.")

if __name__ == "__main__":
    find_vertigo_bounds()

Searching for 10 matches on map 'vertigo'...

[1/10] Processed bounds for match: 002f2b1a-55e2-4b09-9d2e-2d5420b6e3cb
[2/10] Processed bounds for match: 02d6aefa-37ae-4ece-a77b-95c5a45a681b
[3/10] Processed bounds for match: 04557100-a2b0-4904-b81f-240dc22c0681
[4/10] Processed bounds for match: 07d7ac8f-f275-49ef-a87d-0a84681ec114
[5/10] Processed bounds for match: 08f0351b-6b8e-4a04-bf27-5e05245d9238
[6/10] Processed bounds for match: 0c363ab8-49f9-4396-a2b8-6858743407c2
[7/10] Processed bounds for match: 118684ff-1f28-45ba-937a-ef3dcd4014fd
[8/10] Processed bounds for match: 16cd2a38-8b46-4239-8e6e-bfc1b16f58e2
[9/10] Processed bounds for match: 17e24cd0-6532-41ca-9a9e-124429055b07
[10/10] Processed bounds for match: 188b9bb0-3d07-4e66-b44a-c8e56a9f6646

 EMPIRICAL MAP BOUNDS FOR: VERTIGO
 Matches analyzed: 10
 x_min :   -2732.43
 x_max :       5.23
 y_min :   -1619.93
 y_max :    1114.28

Dictionary Output format:
MAP_BOUNDS_MIRAGE = {'x_min': -2732.430908203125, 'x_max': 5.2311325